# 02 — The protocol, and what it says

The first version of this comparison concluded that the semi-supervised arm improved recall
on the cancer class, 0.900 to 0.960. Three leaks stood behind that number, all of them
pushing the same way:

1. **31 of the 99 evaluation images were in the pre-training pool** (notebook 01);
2. the **clustering method** was chosen by ARI against every label, test folds included;
3. the **cluster-to-class alignment** was decided by a vote over those same labels.

And a confound: the semi-supervised arm took strictly more gradient steps than its baseline.

This notebook reads the artefacts of the corrected protocol. It is not there to defend a
conclusion — it is there to report one.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## 1. What the corrected protocol does differently

* Duplicates are removed **by content**, and the evaluation set never loses an image.
* The clustering, the choice of method and the alignment happen **inside the training
  fold**. The test fold takes part in no decision.
* Four arms share folds, architecture and starting weights:

| arm | pre-training | what it isolates |
|---|---|---|
| `supervised` | none | the reference |
| `semi_supervised` | the fold's pseudo-labels | the supposed contribution |
| `semi_supervised_confident` | only the pseudo-labels above a confidence cut | whether noise was the problem |
| `permuted_control` | the same images, labels shuffled | budget and exposure |

* The checkpoint and the decision threshold come from an inner validation split carved out
  of the training fold.
* Five repeats of a five-fold cross-validation, so the spread is measured rather than
  guessed.

The control is the arm that can end the discussion. It sees the same images and takes the
same number of steps; only the pairing between an image and its pseudo-label is destroyed.
Every pre-training arm here has one.

In [ ]:
import json

from mri_semisupervised.config import EXPERIMENTS_DIR

corrected = EXPERIMENTS_DIR / "corrected"
print(f"reading {corrected.name}")

meta = json.loads((corrected / "manifest.json").read_text(encoding="utf-8"))
print(f"dataset fingerprint : {meta['dataset_fingerprint']}")
print(f"evaluation images   : {meta['evaluation_images']}")
print(f"unlabelled pool     : {meta['unlabelled_pool']}")
print(f"folds               : {meta['protocol']['n_splits']} x {meta['protocol']['n_repeats']} repeats")
print(f"gpu                 : {meta['versions']['gpu']}")

per_fold = pd.read_parquet(corrected / "per_fold.parquet")
predictions = pd.read_parquet(corrected / "predictions.parquet")
folds = pd.read_parquet(corrected / "folds.parquet")

## 2. The arms, side by side

Two families of number, and they answer different questions. **ROC AUC and PR-AUC** say how
well the scores rank, whatever threshold is applied. **Recall and F1** say what happens at
the threshold that was actually chosen — on the inner validation, never on the test fold.

The original comparison reported only the second kind, which is how a model whose ranking
had got *worse* came to look better.

In [ ]:
headline = ["roc_auc", "pr_auc", "recall_positive", "f1_macro", "accuracy"]
per_fold.groupby("arm")[headline].agg(["mean", "std"]).round(3)

## 3. Paired comparisons

The arms share their folds, so the comparison is paired. Treating the two as independent
samples would throw the pairing away and widen every interval for nothing.

In [ ]:
from mri_semisupervised.protocol.uncertainty import paired_difference

pivot = per_fold.pivot(index="fold", columns="arm")
rows = []
for metric in ["roc_auc", "pr_auc", "recall_positive"]:
    for a, b in [
        ("semi_supervised", "supervised"),
        ("semi_supervised", "permuted_control"),
        ("semi_supervised_confident", "permuted_control"),
        ("permuted_control", "supervised"),
    ]:
        out = paired_difference(pivot[(metric, a)].to_numpy(), pivot[(metric, b)].to_numpy())
        rows.append({"metric": metric, "comparison": f"{a} - {b}", **out})
pd.DataFrame(rows).round(4)

The line to read first is `semi_supervised - permuted_control`. If its interval spans zero,
then pre-training on pseudo-labels does no better than pre-training on the same images with
those labels shuffled — and whatever the first version measured was budget and exposure, not
the information the clustering had found.

## 3bis. Where the experiment could see anything at all

A null result is only worth reading if the experiment could have shown a gain. The
supervised baseline reaches 0.966 with eight folds out of twenty-five already at 1.000, so
what is left to win is about the size of the noise. The arms were therefore rerun at
smaller labelling budgets, where a semi-supervised method has room to help.

In [ ]:
budgets = {}
for name, size in [("budget-10", 10), ("budget-20", 20), ("budget-40", 40), ("corrected", 59)]:
    path = EXPERIMENTS_DIR / name / "per_fold.parquet"
    if path.exists():
        budgets[size] = pd.read_parquet(path).pivot(index="fold", columns="arm", values="roc_auc")

rows = []
for size, pivot in sorted(budgets.items()):
    row = {"labels": size}
    row.update({arm: round(pivot[arm].mean(), 3) for arm in pivot.columns})
    if {"semi_supervised", "permuted_control"} <= set(pivot.columns):
        out = paired_difference(
            pivot["semi_supervised"].to_numpy(), pivot["permuted_control"].to_numpy()
        )
        row["semi - control"] = f"{out['mean_difference']:+.3f} (p={out['p_value']:.2f})"
    rows.append(row)
pd.DataFrame(rows)

Two things this table says that a single budget could not. The semi-supervised arm never
beats the plain baseline, at any budget. And the permuted control sits *below* that baseline
everywhere — so the pre-training phase costs something on its own, and real pseudo-labels
recover part of that cost without ever turning it into a gain.

## 3ter. Two hypotheses about why, each with its own control

Two explanations for the null result are worth testing rather than asserting. Maybe the
pre-training is simply **forgotten** — 246 steps, then a fine-tuning that converges in two
to four epochs. Maybe the labels come from the **wrong source** — a k-means on ImageNet
embeddings rather than the decision function being optimised.

`semi_supervised_joint` keeps the pseudo-label loss present at every step;
`self_training` builds its labels from its own first pass. Each is read against its own
control, never against the plain baseline.

In [ ]:
stage_b = EXPERIMENTS_DIR / "corrected-stageb" / "per_fold.parquet"
if stage_b.exists():
    mech = pd.read_parquet(stage_b).pivot(index="fold", columns="arm", values="roc_auc")
    joined = pd.concat([pivot_full := per_fold.pivot(index="fold", columns="arm",
                                                     values="roc_auc"), mech], axis=1)
    rows = []
    for arm, control in [("semi_supervised_joint", "joint_permuted_control"),
                         ("self_training", "self_training_control")]:
        against_control = paired_difference(joined[arm].to_numpy(), joined[control].to_numpy())
        against_baseline = paired_difference(joined[arm].to_numpy(),
                                             joined["supervised"].to_numpy())
        rows.append({
            "arm": arm,
            "vs its control": f"{against_control['mean_difference']:+.3f} "
                              f"(p={against_control['p_value']:.3f})",
            "vs supervised": f"{against_baseline['mean_difference']:+.3f} "
                             f"(p={against_baseline['p_value']:.3f})",
        })
    display(pd.DataFrame(rows))

Joint training beats its control significantly and still loses to the baseline. Both facts
are needed: the control sits at 0.922 because training on shuffled labels at every step is
harmful, so the win against it measures that harm rather than a gain. A significant p-value
against the correct control can still mean the opposite of what it looks like.

## 3quater. What a returned probability is worth

The protocol publishes scores. Whether they mean anything as probabilities is a separate
question, and ROC AUC cannot answer it: a model whose ranking is perfect and whose scale is
squashed scores 1.0 either way.

In [ ]:
from mri_semisupervised.protocol.calibration import summarise_calibration

summary, curves = summarise_calibration(predictions, n_bins=10)
display(summary.round(4))

sup = curves["supervised"]
display(sup[["mean_score", "observed", "gap"]].round(3))

Read the `gap` column: where the model predicts 0.44 the observed cancer rate is 0.67. The
mid-range scores understate risk by twenty points and more, which is the direct reason an
operating point chosen on one split does not transport to another.

Nothing is recalibrated here. A network fine-tuned on twenty images per fold has little
chance of being calibrated, and measuring that and saying so is the result.

In [ ]:
from mri_semisupervised.viz.plots import plot_roc_compare

curves = {
    arm: (group["y_true"].to_numpy(), group["y_score"].to_numpy())
    for arm, group in predictions.groupby("arm")
}
plot_roc_compare(curves, title="Pooled out-of-fold ROC, five repeats")

## 4. Which clustering method each fold picked

Choosing the method inside the fold means the choice can differ from one fold to the next.
That is not instability to hide — it is a measurement of how much the choice depended on
seeing every label.

In [ ]:
print(folds["pseudo_method"].value_counts().to_string())
print()
print(f"ARI on the training labels: {folds['pseudo_ari_on_train'].mean():.3f} "
      f"+/- {folds['pseudo_ari_on_train'].std():.3f}")
print(f"pseudo-labels per fold    : {folds['n_pseudo'].mean():.0f}")

## 5. What the leaks were worth

The `legacy` run reproduces the original protocol faithfully — duplicates left in place,
method and alignment decided once over every label, checkpoint taken on training accuracy.
It exists so the difference is **measured** rather than quoted from an old file.

In [ ]:
legacy_dir = EXPERIMENTS_DIR / "legacy"
if (legacy_dir / "per_fold.parquet").exists():
    legacy = pd.read_parquet(legacy_dir / "per_fold.parquet")
    comparison = pd.concat(
        [
            per_fold.groupby("arm")[headline].mean().add_suffix("_corrected"),
            legacy.groupby("arm")[headline].mean().add_suffix("_legacy"),
        ],
        axis=1,
    )
    display(comparison.round(3))
else:
    print("no legacy run found: uv run python scripts/run_experiment.py --mode legacy")

## 6. What to take away

Whatever the numbers above say, the method is the point:

* an identity that comes from the data, so a duplicate cannot hide behind a folder;
* every decision that reads a label made inside the training fold;
* a control arm that makes "it helped" a falsifiable claim rather than a hopeful one;
* intervals, because with twenty images per fold a difference of one image moves recall by
  0.05.

A protocol that can only confirm what you hoped is not a protocol.